# 08 | Entrenamiento y validación

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: ENTRENA — ajusta modelos reales sobre el dataset (puede tardar minutos).**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## Entrenamiento real (escenario A)

Se ajustan los 5 modelos con el módulo del protocolo v2. Primero el protocolo por predios; luego el protocolo espacial independiente.

In [2]:
import sys
sys.path.insert(0, str(REPO / 'ml' / 'src'))
from yacopi_ml.valor.dataset import construir_dataset, seleccionar_features
from yacopi_ml.valor.splits import dividir_predios, dividir_espacial
from yacopi_ml.valor.modelos import entrenar_modelos, predecir
ds = construir_dataset('A')
X, y, _, _ = seleccionar_features(ds)
ds = ds.assign(**{c: X[c] for c in X.columns})
tr, te = dividir_predios(ds)
tr_esp, te_esp = dividir_espacial(ds)
print('train/test predio:', len(tr), '/', len(te))
print('train/test espacial:', len(tr_esp), '/', len(te_esp))
print('solape tr_predio vs te_esp:', len(set(tr.numero_predial) & set(te_esp.numero_predial)))
print('solape tr_esp vs te_esp:', len(set(tr_esp.numero_predial) & set(te_esp.numero_predial)))

train/test predio: 3740 / 936
train/test espacial: 3987 / 689
solape tr_predio vs te_esp: 560
solape tr_esp vs te_esp: 0


In [3]:
modelos = entrenar_modelos(tr[X.columns], tr['valor_terreno_ha'])
preds = predecir(modelos, te[X.columns])
from sklearn.metrics import r2_score
print('PROTOCOLO PREDIO (entrena con tr, evalua con te):')
for nombre, p in preds.items():
    print(f'  {nombre:14s} R2={r2_score(te.valor_terreno_ha, p):+.3f}')

PROTOCOLO PREDIO (entrena con tr, evalua con te):
  dummy_media    R2=-0.000
  arbol          R2=+0.459
  random_forest  R2=+0.689
  ridge_log      R2=+0.302
  catboost       R2=+0.725


## Evidencia de la contaminación v1 (ya corregida en v2)

En la versión v1, el modelo entrenado con `tr` (partición aleatoria) se evaluaba sobre `te_esp`: **560 de 689 predios del test espacial (81,3 %) ya estaban en el entrenamiento**. En v2 el protocolo espacial entrena una instancia NUEVA con `tr_esp` y solo entonces evalúa `te_esp` (solape 0).

In [4]:
modelos_esp = entrenar_modelos(tr_esp[X.columns], tr_esp['valor_terreno_ha'])
preds_esp = predecir(modelos_esp, te_esp[X.columns])
print('PROTOCOLO ESPACIAL INDEPENDIENTE (entrena con tr_esp, evalua con te_esp):')
for nombre, p in preds_esp.items():
    print(f'  {nombre:14s} R2={r2_score(te_esp.valor_terreno_ha, p):+.3f}')

PROTOCOLO ESPACIAL INDEPENDIENTE (entrena con tr_esp, evalua con te_esp):
  dummy_media    R2=-0.006
  arbol          R2=+0.092
  random_forest  R2=+0.171
  ridge_log      R2=+0.011
  catboost       R2=-0.075


In [5]:
import json
r = json.loads((REPO / 'ml/artifacts/exp-valor-terreno-001/resumen.json').read_text(encoding='utf-8'))
for esc in ['A','B']:
    print('== escenario', esc, '==')
    for proto in ['predio','espacial']:
        p = r[esc]['protocolos'][proto]
        print(' --', proto, 'train/test:', p['n_train'], '/', p['n_test'])
        for nombre, res in p['resultados'].items():
            if nombre.startswith('_') or isinstance(res, list): continue
            print(f"    {nombre:16s} MAE={res['mae_cop_ha']:>11.0f} R2={res['r2']:+.3f}")

== escenario A ==
 -- predio train/test: 3740 / 936
    dummy_media      MAE=     758752 R2=-0.000
    arbol            MAE=     343025 R2=+0.459
    random_forest    MAE=     248296 R2=+0.689
    ridge_log        MAE=     392022 R2=+0.302
    catboost         MAE=     279284 R2=+0.725
 -- espacial train/test: 3987 / 689
    dummy_media      MAE=    1237011 R2=-0.006
    arbol            MAE=     805690 R2=+0.092
    random_forest    MAE=     767064 R2=+0.171
    ridge_log        MAE=     838996 R2=+0.011
    catboost         MAE=     897642 R2=-0.075
== escenario B ==
 -- predio train/test: 788 / 198
    dummy_media      MAE=   15347647 R2=-0.015
    arbol            MAE=   11846090 R2=+0.118
    random_forest    MAE=   11487894 R2=+0.146
    ridge_log        MAE=   12726825 R2=+0.046
    catboost         MAE=   11670499 R2=+0.118
 -- espacial train/test: 809 / 177
    dummy_media      MAE=    6175772 R2=-0.087
    arbol            MAE=    1149744 R2=+0.618
    random_forest    MAE=  